# DermDepth: Synthetic Data Generation (Colab)

Generate S-SYNTH training data for MoGe-2 fine-tuning. Produces RGB images + depth maps + camera intrinsics.

**Shard-based**: Set `SHARD_ID` and `NUM_SHARDS` to split work across multiple Colab instances.

**Requirements**: GPU runtime (T4/A100/L4). Assets auto-download from HuggingFace.

## 0. Configuration

Set these before running:

In [ ]:
# === CONFIGURATION ===
SHARD_ID = 0          # This instance's shard (0, 1, 2, ...)
NUM_SHARDS = 1        # Total number of Colab instances
TOTAL_SAMPLES = 2000  # Total dataset size (split across shards)
SPP = 128             # Samples per pixel (paper uses 128)
SEED = 42             # Random seed for reproducible sampling

# HuggingFace token (needed for didsr/ssynth_data, a gated dataset)
HF_TOKEN = ""  # Paste your HF token here, or leave empty to use huggingface-cli login

# Output: Google Drive mount
USE_DRIVE = True       # Mount Google Drive for persistent output
DRIVE_OUTPUT = "/content/drive/MyDrive/dermdepth_train"  # Output path on Drive
LOCAL_OUTPUT = "/content/dermdepth_train"  # Fallback local output

OUTPUT_DIR = DRIVE_OUTPUT if USE_DRIVE else LOCAL_OUTPUT

print(f"Shard {SHARD_ID}/{NUM_SHARDS}, {TOTAL_SAMPLES} total samples, SPP={SPP}")

## 1. Setup Environment

In [ ]:
# Mount Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"Output dir: {DRIVE_OUTPUT}")

In [ ]:
# Install dependencies
!pip install -q mitsuba huggingface_hub pillow numpy matplotlib

# Verify GPU and mitsuba
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

import mitsuba as mi
print(f"Mitsuba version: {mi.__version__}")
print(f"Available variants: {mi.variants()}")

# Set variant - no debug mode needed on Colab (modern drivers)
mi.set_variant('cuda_ad_spectral')
print("CUDA spectral variant loaded successfully!")

In [ ]:
# Clone dermdepth repo and download S-SYNTH assets from HuggingFace
import os, shutil

REPO_DIR = "/content/dermdepth"
SSYNTH_DIR = "/content/ssynth_data"  # Where assets will live

# Clone repo (for reference / any utility scripts)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/ANONYMOUS/dermdepth.git {REPO_DIR}
else:
    print(f"Repo already cloned at {REPO_DIR}")

# Login to HuggingFace if token provided
if HF_TOKEN:
    !huggingface-cli login --token {HF_TOKEN}
else:
    print("No HF_TOKEN set. If download fails, set HF_TOKEN above or run: huggingface-cli login")

# Download assets from HuggingFace
from huggingface_hub import hf_hub_download

os.makedirs(SSYNTH_DIR, exist_ok=True)

for asset_name in ['materials.zip', 'hdri.zip', 'params_lists.zip']:
    target = os.path.join(SSYNTH_DIR, 'data', 'supporting_data', asset_name)
    if not os.path.exists(target):
        print(f"Downloading {asset_name}...")
        hf_hub_download(
            repo_id="didsr/ssynth_data",
            repo_type="dataset",
            local_dir=SSYNTH_DIR,
            filename=f'data/supporting_data/{asset_name}',
            token=HF_TOKEN if HF_TOKEN else True,
        )
    else:
        print(f"{asset_name} already downloaded")

# Unzip assets
support_dir = os.path.join(SSYNTH_DIR, 'data', 'supporting_data')
for zf in ['materials.zip', 'hdri.zip', 'params_lists.zip']:
    zpath = os.path.join(support_dir, zf)
    # Check if already extracted by looking for a key directory
    check_dir = os.path.join(support_dir, zf.replace('.zip', ''))
    if os.path.exists(zpath) and not os.path.exists(check_dir):
        print(f"Extracting {zf}...")
        shutil.unpack_archive(zpath, support_dir, 'zip')
    else:
        print(f"{zf} already extracted or not found")

# Verify assets
print("\nAsset verification:")
for d in ['materials/outputModels', 'materials/opticalMaterials', 'materials/lesions_release', 'hdri']:
    p = os.path.join(support_dir, d)
    if os.path.exists(p):
        n = len(os.listdir(p))
        print(f"  {d}: {n} files")
    else:
        print(f"  {d}: MISSING!")

print("\nSetup complete!")

## 2. Rendering Code

All rendering logic embedded below (self-contained, no external imports needed).

In [ ]:
# === CONFIG: Asset paths ===
ASSET_BASE = os.path.join(SSYNTH_DIR, 'data', 'supporting_data')
sDir = os.path.join(ASSET_BASE, 'materials/')  
sDir_lesion_ver0 = os.path.join(ASSET_BASE, 'materials/lesions_release/ver0/')
sDir_lesion_ver1 = os.path.join(ASSET_BASE, 'materials/lesions_release/ver1/')
sDir_hdri = os.path.join(ASSET_BASE, 'hdri/')

print(f"Asset base: {ASSET_BASE}")
print(f"Materials: {sDir}")
print(f"HDRI: {sDir_hdri}")

In [ ]:
# === DEPTH UTILITIES (depth_utils.py) ===

import numpy as np
import json
from PIL import Image, PngImagePlugin


def compute_intrinsics(fov_deg=75, width=1024, height=1024):
    """Compute camera intrinsics matrix from FOV and resolution."""
    fov_rad = np.deg2rad(fov_deg)
    fx = (width / 2) / np.tan(fov_rad / 2)
    fy = (height / 2) / np.tan(fov_rad / 2)
    cx = width / 2
    cy = height / 2
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float32)


def normalize_intrinsics(K, width, height):
    """Normalize pixel-space intrinsics to [0,1] range expected by MoGe."""
    K_norm = K.copy()
    K_norm[0, 0] /= width
    K_norm[0, 2] /= width
    K_norm[1, 1] /= height
    K_norm[1, 2] /= height
    return K_norm


def save_depth_moge(path, depth_array, camera_height=None):
    """Save depth in MoGe-compatible 16-bit PNG with logarithmic encoding."""
    depth = depth_array.astype(np.float32)
    mask_valid = np.isfinite(depth) & (depth > 0)
    if not mask_valid.any():
        raise ValueError("No valid depth values in depth array")

    near = max(depth[mask_valid].min(), 1e-5)
    far = max(near * 1.1, depth[mask_valid].max())

    depth_norm = (np.log(np.clip(depth, near, far) / near) / np.log(far / near))
    depth_encoded = (1 + np.round(depth_norm.clip(0, 1) * 65533)).astype(np.uint16)
    depth_encoded[~mask_valid] = 0
    depth_encoded[np.isinf(depth_array) & (depth_array > 0)] = 65535

    pil_image = Image.fromarray(depth_encoded)
    pnginfo = PngImagePlugin.PngInfo()
    pnginfo.add_text('near', str(near))
    pnginfo.add_text('far', str(far))
    if camera_height is not None:
        pnginfo.add_text('camera_height', str(camera_height))
    pil_image.save(path, pnginfo=pnginfo, compress_level=7)


def load_depth_moge(path):
    """Load depth from MoGe-format 16-bit PNG with logarithmic decoding."""
    pil_image = Image.open(path)
    depth_encoded = np.array(pil_image, dtype=np.uint16)
    metadata = {}
    if 'near' in pil_image.info:
        metadata['near'] = float(pil_image.info['near'])
    if 'far' in pil_image.info:
        metadata['far'] = float(pil_image.info['far'])
    near = metadata.get('near', 1e-5)
    far = metadata.get('far', 100.0)
    depth_norm = (depth_encoded.astype(np.float32) - 1) / 65533.0
    depth = near * np.exp(depth_norm * np.log(far / near))
    depth[depth_encoded == 0] = np.nan
    depth[depth_encoded == 65535] = np.inf
    return depth, metadata


def save_meta_json(path, fov_deg=75, width=1024, height=1024, additional_meta=None):
    """Save camera intrinsics and metadata in MoGe-compatible JSON format."""
    intrinsics_pixel = compute_intrinsics(fov_deg, width, height)
    intrinsics_norm = normalize_intrinsics(intrinsics_pixel, width, height)
    meta = {
        'intrinsics': intrinsics_norm.tolist(),
        'fov_deg': fov_deg,
        'width': width,
        'height': height
    }
    if additional_meta:
        meta.update(additional_meta)
    with open(path, 'w') as f:
        json.dump(meta, f, indent=2, default=lambda x: int(x) if isinstance(x, np.integer) else
                  float(x) if isinstance(x, np.floating) else str(x))


def validate_depth(depth_array, expected_min=5.0, expected_max=50.0):
    """Validate depth array values."""
    valid_mask = np.isfinite(depth_array) & (depth_array > 0)
    valid_depth = depth_array[valid_mask]
    results = {
        'valid_pixels': int(valid_mask.sum()),
        'total_pixels': depth_array.size,
        'valid_ratio': float(valid_mask.sum() / depth_array.size),
        'warnings': []
    }
    if len(valid_depth) > 0:
        results['min'] = float(valid_depth.min())
        results['max'] = float(valid_depth.max())
        results['mean'] = float(valid_depth.mean())
        results['std'] = float(valid_depth.std())
    return results


print("Depth utilities loaded.")

In [ ]:
# === RENDERING FUNCTIONS (render_extended_features.py) ===

import mitsuba as mi
import time
import math


def ray_distance_to_z_depth(ray_distance, fov_deg, width=1024, height=1024):
    """Convert ray distance to Z-depth (perpendicular distance from camera plane)."""
    fov_rad = np.deg2rad(fov_deg)
    focal = (width / 2) / np.tan(fov_rad / 2)
    u = np.arange(width, dtype=np.float32)
    v = np.arange(height, dtype=np.float32)
    u_grid, v_grid = np.meshgrid(u, v)
    cx, cy = width / 2, height / 2
    dx = (u_grid - cx) / focal
    dy = (v_grid - cy) / focal
    cos_theta = 1.0 / np.sqrt(1.0 + dx**2 + dy**2)
    return ray_distance * cos_theta


def get_sensor(camera_height=15, fov=75, width=1024, height=1024, angle_offset=(0, 0)):
    """Create camera sensor for RGB rendering."""
    x_off = camera_height * math.tan(math.radians(angle_offset[0]))
    z_off = camera_height * math.tan(math.radians(angle_offset[1]))
    origin = [x_off, camera_height, z_off]
    return mi.load_dict({
        'type': 'perspective',
        'to_world': mi.ScalarTransform4f.look_at(
            target=[0, 0, 0], origin=origin, up=[0, 0, 1]
        ),
        'fov': fov,
        'film': {'type': 'hdrfilm', 'width': width, 'height': height}
    })


def create_multi_lesion_scene(model_id, lesion_configs, melanin, blood_frac,
                               light_name, hair_model=-1, skin_scale=1.0):
    """Create a scene with multiple lesions for spectral volumetric rendering."""
    uniform_scale = skin_scale
    y_offset = -1.5 * skin_scale
    room_size = 20
    xt_scale = 0.1

    ext_hair = 28.5 + 37.5
    hair_albedo = [0.84, 0.6328, 0.44]

    ior_epi = 1.43
    ior_blood = 1.36
    ior_hypo = 1.44
    A, B, C = 1.3696, 3916.8, 2558.8
    ior_derm = A + (B / (500 ** 2)) + (C / (500 ** 4))

    # Adjust optical thickness for scaled skin
    xt_adj = xt_scale / skin_scale if skin_scale != 1.0 else xt_scale

    scene = {
        'type': 'scene',
        'integrator': {'type': 'volpathmis', 'max_depth': 50}
    }

    # Lesions
    for i, lcfg in enumerate(lesion_configs):
        version = lcfg.get('version', 'ver1')
        lesion_dir = sDir_lesion_ver0 if version == 'ver0' else sDir_lesion_ver1
        lesion_file = f"{lesion_dir}/lesion{lcfg['lesion_id']}_T{lcfg['time_point']:03d}.obj"
        pos_x, pos_z = lcfg.get('position', (0, 0))
        scale = lcfg.get('scale', 1.5)
        lesion_offset = lcfg.get('y_offset', -2)

        scene[f'lesion_{i}'] = {
            'type': 'obj',
            'filename': lesion_file,
            'to_world': mi.ScalarTransform4f.scale(scale).translate([pos_x, lesion_offset, pos_z]),
            'bsdf': {
                'type': 'roughdielectric', 'alpha': 0.01,
                'int_ior': 1.32988 - (-3.97577e7) * 0.95902 ** 500,
                'ext_ior': 1.000277
            },
            'interior': {
                'type': 'homogeneous',
                'albedo': {'type': 'spectrum', 'filename': sDir + f"opticalMaterials/{lcfg['lesion_mat']}_alb.spd"},
                'sigma_t': {'type': 'spectrum', 'filename': sDir + f"opticalMaterials/{lcfg['lesion_mat']}_ext.spd"},
                'scale': xt_scale
            }
        }

    # Hair
    if hair_model >= 0:
        scene['hair'] = {
            'type': 'obj',
            'filename': sDir + f'outputModels/hair_{hair_model:03d}.obj',
            'to_world': mi.ScalarTransform4f.scale(uniform_scale).translate([0, y_offset / skin_scale, 0]),
            'bsdf': {
                'type': 'roughdielectric', 'alpha': 0.01,
                'int_ior': 1.55, 'ext_ior': 1.000277
            },
            'interior': {
                'type': 'homogeneous',
                'sigma_t': ext_hair,
                'albedo': {'type': 'rgb', 'value': hair_albedo},
                'scale': 3
            }
        }

    # Skin layers
    for layer, ior, alb_file, ext_file in [
        ('epidermis', ior_epi,
         f'opticalMaterials/epidermis_alb_mel{melanin}.spd',
         f'opticalMaterials/epidermis_ext_mel{melanin}.spd'),
        ('vascular', ior_blood,
         'opticalMaterials/blood_HbO2_alb.spd',
         'opticalMaterials/blood_HbO2_ext.spd'),
        ('dermis', ior_derm,
         f'opticalMaterials/dermis_alb_fB{blood_frac}.spd',
         f'opticalMaterials/dermis_ext_fB{blood_frac}.spd'),
        ('hypodermis', ior_hypo,
         'opticalMaterials/hypo_alb.spd',
         'opticalMaterials/hypo_ext.spd'),
    ]:
        scene[layer] = {
            'type': 'obj',
            'filename': sDir + f'outputModels/{layer}_{model_id:03d}.obj',
            'to_world': mi.ScalarTransform4f.scale(uniform_scale).translate([0, y_offset / skin_scale, 0]),
            'bsdf': {
                'type': 'roughdielectric', 'alpha': 0.01,
                'int_ior': ior, 'ext_ior': 1.000277
            },
            'interior': {
                'type': 'homogeneous',
                'albedo': {'type': 'spectrum', 'filename': sDir + alb_file},
                'sigma_t': {'type': 'spectrum', 'filename': sDir + ext_file},
                'scale': xt_adj
            }
        }

    # Environment lighting
    scene['env_light'] = {
        'type': 'envmap',
        'filename': sDir_hdri + '/' + light_name + '.exr',
        'scale': 3
    }

    # Floor
    scene['wall_floor'] = {
        'type': 'rectangle',
        'to_world': mi.ScalarTransform4f.scale([room_size, 1, room_size]).translate([0, -room_size, 0]).rotate([1, 0, 0], -90),
        'bsdf': {
            'type': 'twosided',
            'material': {'type': 'diffuse', 'reflectance': {'type': 'rgb', 'value': 0.5}}
        }
    }

    return scene


def render_depth_proper(model_id, lesion_configs, camera_height=15, fov=75,
                        angle_offset=(0, 0), skin_scale=1.0, hair_model=-1):
    """Render depth using AOV integrator and convert to Z-depth. Skin/lesion only, no floor."""
    y_offset = -1.5 * skin_scale

    scene_dict = {
        'type': 'scene',
        'integrator': {'type': 'aov', 'aovs': 'dd.y:depth', 'nested': {'type': 'direct'}}
    }

    simple_bsdf = {'type': 'diffuse', 'reflectance': {'type': 'rgb', 'value': [0.5, 0.5, 0.5]}}

    # Epidermis
    scene_dict['epidermis'] = {
        'type': 'obj',
        'filename': sDir + f'outputModels/epidermis_{model_id:03d}.obj',
        'to_world': mi.ScalarTransform4f.scale(skin_scale).translate([0, y_offset / skin_scale, 0]),
        'bsdf': simple_bsdf
    }

    # Hair
    if hair_model >= 0:
        scene_dict['hair'] = {
            'type': 'obj',
            'filename': sDir + f'outputModels/hair_{hair_model:03d}.obj',
            'to_world': mi.ScalarTransform4f.scale(skin_scale).translate([0, y_offset / skin_scale, 0]),
            'bsdf': simple_bsdf
        }

    # Lesions
    for i, lcfg in enumerate(lesion_configs):
        version = lcfg.get('version', 'ver1')
        lesion_dir = sDir_lesion_ver0 if version == 'ver0' else sDir_lesion_ver1
        lesion_file = f"{lesion_dir}/lesion{lcfg['lesion_id']}_T{lcfg['time_point']:03d}.obj"
        pos_x, pos_z = lcfg.get('position', (0, 0))
        scale = lcfg.get('scale', 1.5)
        lesion_offset = lcfg.get('y_offset', -2)
        scene_dict[f'lesion_{i}'] = {
            'type': 'obj', 'filename': lesion_file,
            'to_world': mi.ScalarTransform4f.scale(scale).translate([pos_x, lesion_offset, pos_z]),
            'bsdf': simple_bsdf
        }

    # Light
    scene_dict['light'] = {
        'type': 'directional', 'direction': [0, -1, 0],
        'irradiance': {'type': 'rgb', 'value': 1.0}
    }

    # Sensor
    x_off = camera_height * math.tan(math.radians(angle_offset[0]))
    z_off = camera_height * math.tan(math.radians(angle_offset[1]))
    width, height = 1024, 1024
    sensor = mi.load_dict({
        'type': 'perspective',
        'to_world': mi.ScalarTransform4f.look_at(
            target=[0, 0, 0], origin=[x_off, camera_height, z_off], up=[0, 0, 1]
        ),
        'fov': fov,
        'film': {'type': 'hdrfilm', 'width': width, 'height': height, 'pixel_format': 'rgba'}
    })

    scene = mi.load_dict(scene_dict)
    image = mi.render(scene, sensor=sensor, spp=1)
    depth_array = np.array(image)

    # Extract depth AOV channel
    if depth_array.ndim == 3:
        if depth_array.shape[2] >= 5:
            ray_distance = depth_array[:, :, 4]
        elif depth_array.shape[2] >= 4:
            ch3 = depth_array[:, :, 3]
            if ch3.max() - ch3.min() > 0.1:
                ray_distance = ch3
            else:
                ray_distance = depth_array[:, :, 0]
        else:
            ray_distance = depth_array[:, :, 0]
    else:
        ray_distance = depth_array

    ray_distance = np.where(ray_distance < 0.1, np.nan, ray_distance)
    z_depth = ray_distance_to_z_depth(ray_distance, fov, width, height)
    return z_depth


def render_sample_with_depth(model_id, lesion_configs, melanin, blood_frac,
                              light_name, output_dir, sample_name,
                              camera_height=15, fov=75, angle_offset=(0, 0),
                              skin_scale=1.0, hair_model=-1, spp=128):
    """Render complete sample: RGB + depth + metadata."""
    os.makedirs(output_dir, exist_ok=True)
    result = {
        'name': sample_name,
        'camera_height': camera_height,
        'angle_offset': angle_offset,
        'skin_scale': skin_scale
    }

    # === RGB ===
    print(f"    Rendering RGB (spp={spp})...")
    mi.set_variant('cuda_ad_spectral')

    scene_dict = create_multi_lesion_scene(
        model_id, lesion_configs, melanin, blood_frac, light_name,
        hair_model, skin_scale
    )
    sensor = get_sensor(camera_height, fov, angle_offset=angle_offset)

    start = time.time()
    scene = mi.load_dict(scene_dict)
    image = mi.render(scene, sensor=sensor, spp=spp)
    rgb_time = time.time() - start

    image_array = np.array(image)[:, :, :3]
    image_uint8 = (np.clip(image_array, 0, 1) * 255).astype(np.uint8)
    rgb_path = os.path.join(output_dir, f"{sample_name}_rgb.png")
    Image.fromarray(image_uint8).save(rgb_path)
    result['rgb_path'] = rgb_path
    result['rgb_time'] = rgb_time

    # === Depth ===
    print(f"    Rendering depth...")
    mi.set_variant('scalar_rgb')

    start = time.time()
    depth = render_depth_proper(
        model_id, lesion_configs,
        camera_height=camera_height, fov=fov,
        angle_offset=angle_offset, skin_scale=skin_scale,
        hair_model=hair_model
    )
    depth_time = time.time() - start

    depth_path = os.path.join(output_dir, f"{sample_name}_depth.png")
    save_depth_moge(depth_path, depth, camera_height)
    result['depth_path'] = depth_path
    result['depth_time'] = depth_time

    validation = validate_depth(depth)
    result['depth_validation'] = validation

    # === Metadata ===
    meta_path = os.path.join(output_dir, f"{sample_name}_meta.json")
    save_meta_json(
        meta_path, fov_deg=fov, width=1024, height=1024,
        additional_meta={
            'camera_height_mm': camera_height,
            'angle_offset_deg': angle_offset,
            'skin_scale': skin_scale,
            'depth_range_mm': [validation.get('min', 0), validation.get('max', 0)],
            'lesion_configs': lesion_configs,
            'melanin': melanin,
            'blood_fraction': blood_frac
        }
    )
    result['meta_path'] = meta_path

    # Switch back
    mi.set_variant('cuda_ad_spectral')
    return result


print("Rendering functions loaded.")

In [ ]:
# === DATASET GENERATION (generate_dermdepth_dataset.py) ===

import random

# Melanin levels (131 string values matching S-SYNTH .spd filenames)
MELANIN_LEVELS = [
    '0.005', '0.01', '0.015', '0.02', '0.025', '0.03', '0.031', '0.033',
    '0.035', '0.039', '0.04', '0.041', '0.042', '0.05', '0.06', '0.061',
    '0.07', '0.08', '0.09', '0.1', '0.11', '0.111', '0.116', '0.12',
    '0.123', '0.13', '0.14', '0.141', '0.142', '0.144', '0.15', '0.153',
    '0.155', '0.159', '0.16', '0.17', '0.18', '0.19', '0.2', '0.207',
    '0.21', '0.217', '0.22', '0.23', '0.24', '0.25', '0.254', '0.258',
    '0.26', '0.27', '0.28', '0.29', '0.297', '0.3', '0.31', '0.32',
    '0.33', '0.34', '0.341', '0.35', '0.36', '0.37', '0.38', '0.387',
    '0.39', '0.391', '0.4', '0.41', '0.42', '0.424', '0.43', '0.44',
    '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52',
    '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.6',
    '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68',
    '0.69', '0.7', '0.71', '0.72', '0.73', '0.74', '0.75', '0.76',
    '0.77', '0.78', '0.79', '0.8', '0.81', '0.82', '0.83', '0.84',
    '0.85', '0.86', '0.87', '0.88', '0.89', '0.9', '0.91', '0.92',
    '0.93', '0.94', '0.95', '0.96', '0.97', '0.98', '0.99', '1.0',
    '1.1', '1.5', '2.0',
]

FITZPATRICK_GROUPS = {
    'I-II':   [m for m in MELANIN_LEVELS if float(m) <= 0.05],
    'III-IV': [m for m in MELANIN_LEVELS if 0.05 < float(m) <= 0.25],
    'V-VI':   [m for m in MELANIN_LEVELS if float(m) > 0.25],
}

BLOOD_FRACTIONS = [0.002, 0.005, 0.02, 0.05]
SKIN_MODELS = list(range(0, 100))
LESION_IDS_VER1 = list(range(1, 21))
LESION_TIMEPOINTS_VER1 = [2, 5, 10, 15, 20, 25, 30]
LESION_IDS_VER0 = list(range(1, 21))
LESION_TIMEPOINTS_VER0 = [10, 20, 30, 40, 50]

LESION_MATERIALS = [
    'HbO2x0.1Epix0.025', 'HbO2x0.1Epix0.05', 'HbO2x0.1Epix0.1',
    'HbO2x0.1Epix0.15', 'HbO2x0.1Epix0.25', 'HbO2x0.1Epix0.4',
    'HbO2x0.5Epix0.025', 'HbO2x0.5Epix0.05', 'HbO2x0.5Epix0.1',
    'HbO2x0.5Epix0.15', 'HbO2x0.5Epix0.25', 'HbO2x0.5Epix0.4',
    'HbO2x1.0Epix0.025', 'HbO2x1.0Epix0.05', 'HbO2x1.0Epix0.1',
    'HbO2x1.0Epix0.15', 'HbO2x1.0Epix0.25', 'HbO2x1.0Epix0.4',
]

HDRI_LIGHTS = [
    'bathroom_4k', 'bush_restaurant_4k', 'comfy_cafe_4k',
    'floral_tent_4k', 'graffiti_shelter_4k', 'hospital_room_4k',
    'kiara_interior_4k', 'lapa_4k', 'lythwood_room_4k',
    'reading_room_4k', 'reinforced_concrete_01_4k',
    'rural_asphalt_road_4k', 'school_hall_4k', 'st_fagans_interior_4k',
    'surgery_4k', 'veranda_4k', 'vintage_measuring_lab_4k',
    'vulture_hide_4k', 'yaris_interior_garage_4k',
]

MAX_TILT_DEG = 10  # Tight tilt to avoid layer separation artifacts


def sample_lesion_config(rng):
    """Sample a random lesion configuration."""
    version = rng.choice(['ver0', 'ver1'])
    if version == 'ver1':
        lesion_id = rng.choice(LESION_IDS_VER1)
        time_point = rng.choice(LESION_TIMEPOINTS_VER1)
    else:
        lesion_id = rng.choice(LESION_IDS_VER0)
        time_point = rng.choice(LESION_TIMEPOINTS_VER0)
    return {
        'lesion_id': lesion_id,
        'time_point': time_point,
        'lesion_mat': rng.choice(LESION_MATERIALS),
        'scale': rng.uniform(0.8, 2.0),
        'position': (rng.uniform(-5, 5), rng.uniform(-5, 5)),
        'version': version,
        'y_offset': -2,
    }


def sample_parameters(num_samples, seed=42):
    """Generate stratified parameter combinations (equal across Fitzpatrick groups)."""
    rng = np.random.default_rng(seed)
    samples = []
    samples_per_group = num_samples // len(FITZPATRICK_GROUPS)
    remainder = num_samples % len(FITZPATRICK_GROUPS)

    for group_idx, (group_name, melanin_range) in enumerate(FITZPATRICK_GROUPS.items()):
        n = samples_per_group + (1 if group_idx < remainder else 0)
        for i in range(n):
            melanin = rng.choice(melanin_range)
            blood_frac = rng.choice(BLOOD_FRACTIONS)
            model_id = rng.choice(SKIN_MODELS)
            light_name = rng.choice(HDRI_LIGHTS)
            tilt_mag = float(rng.beta(2, 5) * MAX_TILT_DEG)
            tilt_dir = float(rng.uniform(0, 360))
            tilt_x = tilt_mag * np.cos(np.radians(tilt_dir))
            tilt_z = tilt_mag * np.sin(np.radians(tilt_dir))
            angle = (round(tilt_x, 2), round(tilt_z, 2))
            skin_scale = rng.uniform(1.0, 1.5)  # Larger skin to fill frame
            hair_model = -1 if rng.random() < 0.25 else int(rng.integers(0, 100))
            num_lesions = rng.choice([1, 1, 2, 2, 3])
            lesion_configs = [sample_lesion_config(rng) for _ in range(num_lesions)]
            lesion_configs[0]['position'] = (0, 0)
            camera_height = rng.uniform(12, 16)  # Closer camera for full skin coverage

            samples.append({
                'sample_id': len(samples),
                'fitzpatrick_group': group_name,
                'melanin': melanin,
                'blood_frac': blood_frac,
                'model_id': model_id,
                'light_name': light_name,
                'angle_offset': angle,
                'skin_scale': skin_scale,
                'hair_model': hair_model,
                'camera_height': camera_height,
                'lesion_configs': lesion_configs,
                'spp': SPP,
            })

    rng.shuffle(samples)
    return samples


def render_single_sample(params, output_dir, fov=75):
    """Render a single sample. Returns status dict."""
    sample_id = params['sample_id']
    sample_name = f"sample_{sample_id:06d}"
    sample_dir = os.path.join(output_dir, sample_name)

    # Skip if already rendered
    if os.path.exists(os.path.join(sample_dir, "image.jpg")) and \
       os.path.exists(os.path.join(sample_dir, "depth.png")):
        return {'sample_id': sample_id, 'status': 'skipped', 'name': sample_name}

    try:
        result = render_sample_with_depth(
            model_id=params['model_id'],
            lesion_configs=params['lesion_configs'],
            melanin=params['melanin'],
            blood_frac=params['blood_frac'],
            light_name=params['light_name'],
            output_dir=sample_dir,
            sample_name="render",
            camera_height=params['camera_height'],
            fov=fov,
            angle_offset=tuple(params['angle_offset']),
            skin_scale=params['skin_scale'],
            hair_model=params['hair_model'],
            spp=params['spp'],
        )

        # Convert RGB to JPEG and rename files to MoGe-compatible names
        rgb_src = os.path.join(sample_dir, "render_rgb.png")
        rgb_dst = os.path.join(sample_dir, "image.jpg")
        if os.path.exists(rgb_src) and not os.path.exists(rgb_dst):
            Image.open(rgb_src).save(rgb_dst, quality=95)
            os.remove(rgb_src)
        for src_name, dst_name in [
            ("render_depth.png", "depth.png"),
            ("render_meta.json", "meta.json"),
        ]:
            src = os.path.join(sample_dir, src_name)
            dst = os.path.join(sample_dir, dst_name)
            if os.path.exists(src) and not os.path.exists(dst):
                os.rename(src, dst)

        # Save generation parameters
        params_clean = {}
        for k, v in params.items():
            if isinstance(v, (np.integer,)):
                params_clean[k] = int(v)
            elif isinstance(v, (np.floating,)):
                params_clean[k] = float(v)
            elif isinstance(v, (tuple, list)):
                params_clean[k] = [float(x) if isinstance(x, (np.floating,)) else
                                   int(x) if isinstance(x, (np.integer,)) else x for x in v]
            else:
                params_clean[k] = v
        params_clean['angle_offset'] = list(params['angle_offset'])
        with open(os.path.join(sample_dir, "generation_params.json"), 'w') as f:
            json.dump(params_clean, f, indent=2, default=str)

        return {
            'sample_id': sample_id,
            'status': 'success',
            'name': sample_name,
            'depth_validation': result.get('depth_validation', {}),
        }
    except Exception as e:
        import traceback
        traceback.print_exc()
        return {
            'sample_id': sample_id,
            'status': 'error',
            'name': sample_name,
            'error': str(e),
        }


print("Dataset generation functions loaded.")

## 3. Test Render (1 sample)

Quick sanity check before running full generation.

In [ ]:
import matplotlib.pyplot as plt

# Generate all parameters (deterministic across shards)
all_params = sample_parameters(TOTAL_SAMPLES, seed=SEED)

# Test with first sample from our shard
shard_indices = list(range(SHARD_ID, TOTAL_SAMPLES, NUM_SHARDS))
test_params = all_params[shard_indices[0]]

print(f"Test sample: {test_params['sample_id']}")
print(f"  Fitzpatrick: {test_params['fitzpatrick_group']}")
print(f"  Melanin: {test_params['melanin']}, Blood: {test_params['blood_frac']}")
print(f"  Model: {test_params['model_id']}, Light: {test_params['light_name']}")
print(f"  Camera height: {test_params['camera_height']:.1f}mm")
print(f"  Tilt: {test_params['angle_offset']}")
print(f"  Lesions: {len(test_params['lesion_configs'])}")

# Render test sample
test_dir = "/content/test_render"
result = render_single_sample(test_params, test_dir, fov=75)
print(f"\nResult: {result['status']}")

if result['status'] == 'success':
    sample_dir = os.path.join(test_dir, result['name'])
    rgb = np.array(Image.open(os.path.join(sample_dir, 'image.jpg')))
    depth, meta = load_depth_moge(os.path.join(sample_dir, 'depth.png'))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(rgb)
    axes[0].set_title(f"RGB ({rgb.shape[0]}x{rgb.shape[1]})")
    axes[0].axis('off')

    valid = np.isfinite(depth) & (depth > 0)
    im = axes[1].imshow(depth, cmap='viridis', vmin=depth[valid].min(), vmax=depth[valid].max())
    axes[1].set_title(f"Depth ({depth[valid].min():.1f}-{depth[valid].max():.1f}mm)")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], label='mm')

    # Overlay: depth boundary on RGB
    from scipy import ndimage
    overlay = rgb.copy().astype(np.float32)
    overlay[~valid] *= 0.3
    edges = ndimage.sobel(valid.astype(float))
    overlay[np.abs(edges) > 0.5] = [255, 255, 0]
    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title(f"Overlay (coverage: {valid.mean()*100:.1f}%)")
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()
    print("Test render OK!")
else:
    print(f"Test render FAILED: {result.get('error', 'unknown')}")

## 4. Run Generation (Shard)

Generates this shard's portion of the dataset. Supports resume (skips existing samples).

In [ ]:
# Compute shard assignment
all_params = sample_parameters(TOTAL_SAMPLES, seed=SEED)
shard_indices = list(range(SHARD_ID, TOTAL_SAMPLES, NUM_SHARDS))
shard_params = [all_params[i] for i in shard_indices]

print(f"Shard {SHARD_ID}/{NUM_SHARDS}: {len(shard_params)} samples")
print(f"Sample IDs: {[p['sample_id'] for p in shard_params[:10]]}...")
print(f"Output: {OUTPUT_DIR}")

# Check existing
existing = 0
for p in shard_params:
    d = os.path.join(OUTPUT_DIR, f"sample_{p['sample_id']:06d}")
    if os.path.exists(os.path.join(d, 'image.png')) and os.path.exists(os.path.join(d, 'depth.png')):
        existing += 1
print(f"Already completed: {existing}/{len(shard_params)}")
print(f"Remaining: {len(shard_params) - existing}")

In [ ]:
# === RUN GENERATION ===
os.makedirs(OUTPUT_DIR, exist_ok=True)

start_time = time.time()
results = []
n_total = len(shard_params)

for i, params in enumerate(shard_params):
    result = render_single_sample(params, OUTPUT_DIR, fov=75)
    results.append(result)

    elapsed = time.time() - start_time
    n_done = i + 1
    rate = n_done / (elapsed / 60) if elapsed > 0 else 0
    eta_min = (n_total - n_done) / rate if rate > 0 else 0

    status = result['status']
    if status == 'success':
        v = result.get('depth_validation', {})
        depth_info = f"depth={v.get('min',0):.1f}-{v.get('max',0):.1f}mm"
    else:
        depth_info = status

    print(f"  [{n_done}/{n_total}] {result['name']}: {status} | "
          f"{depth_info} | {rate:.2f}/min | ETA: {eta_min:.0f}min")

# Summary
total_time = time.time() - start_time
success = sum(1 for r in results if r['status'] == 'success')
skipped = sum(1 for r in results if r['status'] == 'skipped')
errors = sum(1 for r in results if r['status'] == 'error')

print(f"\n{'='*50}")
print(f"Shard {SHARD_ID} complete!")
print(f"  Success: {success}, Skipped: {skipped}, Errors: {errors}")
print(f"  Time: {total_time/60:.1f} min ({total_time/3600:.1f} hrs)")
print(f"  Rate: {(success+skipped)/(total_time/60):.2f} samples/min")

# Save shard summary
summary = {
    'shard_id': SHARD_ID,
    'num_shards': NUM_SHARDS,
    'total_samples': TOTAL_SAMPLES,
    'shard_samples': n_total,
    'success': success,
    'skipped': skipped,
    'errors': errors,
    'total_time_min': total_time / 60,
    'error_details': [r for r in results if r['status'] == 'error'],
}
with open(os.path.join(OUTPUT_DIR, f'shard_{SHARD_ID}_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSummary saved to {OUTPUT_DIR}/shard_{SHARD_ID}_summary.json")

## 5. Create MoGe Index File

Run this after ALL shards are complete to create the `.index.txt` for MoGe training.

In [ ]:
# Scan output directory for completed samples
completed = []
for d in sorted(os.listdir(OUTPUT_DIR)):
    sample_dir = os.path.join(OUTPUT_DIR, d)
    if not os.path.isdir(sample_dir) or not d.startswith('sample_'):
        continue
    if os.path.exists(os.path.join(sample_dir, 'image.jpg')) and \
       os.path.exists(os.path.join(sample_dir, 'depth.png')) and \
       os.path.exists(os.path.join(sample_dir, 'meta.json')):
        completed.append(d)

# Write index
index_path = os.path.join(OUTPUT_DIR, '.index.txt')
with open(index_path, 'w') as f:
    f.write('\n'.join(completed))

print(f"Index: {len(completed)} completed samples")
print(f"Written to: {index_path}")

# Check shard summaries
for f_name in sorted(os.listdir(OUTPUT_DIR)):
    if f_name.startswith('shard_') and f_name.endswith('_summary.json'):
        with open(os.path.join(OUTPUT_DIR, f_name)) as f:
            s = json.load(f)
        print(f"  {f_name}: {s['success']} success, {s['errors']} errors, {s['total_time_min']:.0f}min")